In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import os
import json
import warnings
warnings.filterwarnings("ignore")

In [18]:
with open("../parameters/flowParams.json") as f:
    flowParams = json.load(f)

Mach_inf = flowParams["Mach_inf"]
rho_inf = flowParams["rho_inf"]
p_inf = flowParams["p_inf"]

with open("../parameters/bodyParams.json") as f:
    bodyParams = json.load(f)

transitionPoint = bodyParams["transitionPoint"]
bodyLength = bodyParams["bodyLength"]
bodyType = bodyParams["bodyType"]

with open("../parameters/NumericalParams.json") as f:
    NumericalParams = json.load(f)

(
    NumericalParams["N"],
    NumericalParams["M"]
)

(100, 300)

In [19]:
def r_b(z):
    if bodyType == "Cylindrical":
        return np.tan(np.pi / 12)
    elif bodyType == "Cone":
        return np.tan(np.pi / 12) * z
    elif bodyType == "Parabolic":
        return np.tan(np.pi / 12) * np.sqrt(2*z - 1)
    elif bodyType == "DoubleCone":
        double_cone_k = 0.01
        return np.tan(np.pi / 12) + double_cone_k * (z - 1)

In [22]:
def delta_by_C_norm(a, b):
    return np.max(np.abs(a - b))

In [26]:
def delta_by_L1_norm(a, b, r_s, z):
    N, M = a.shape
    dr = (r_s - r_b(z)) / N
    dtheta = np.pi / M
    r = np.zeros_like(a)
    for i in range(N):
        for j in range(M):
            r[i, j] = r_b(z) + i * dr[j]
    return np.sum(np.abs(a - b) * r * dr[np.newaxis, :] * dtheta)

### Сходимость по сетке для автомодельной задачи обтекания конуса (начальное условие основной задачи)

In [3]:
# φ и p(φ), ρ(φ), V_R(φ), V_θ(φ)
initial_data_root = "../initial_cone_flow/output_results/"
data_p   = np.loadtxt(initial_data_root + 'p_init.txt')            # shape (L,2): [φ, p]
data_rho = np.loadtxt(initial_data_root + 'rho_init.txt')          # shape (L,2): [φ, ρ]
data_vr_vtheta = np.loadtxt(initial_data_root + 'VR_Vtheta_init.txt')

phi_cone, p_cone = data_p[::-1,0], data_p[::-1,1]
_, rho_cone = data_rho[::-1,0], data_rho[::-1,1]
_, VR_cone, Vtheta_cone = data_vr_vtheta[::-1,0], data_vr_vtheta[::-1,1], data_vr_vtheta[::-1,2]

### Сходимость по сетке для основной задачи

In [11]:
data_dict = {}

In [40]:
for N_nodes_count in [100, 200, 400, 800]:
    data_root = f"N{N_nodes_count}/"
    
    zs = np.loadtxt(data_root + 'z_out.txt')
    rs = np.loadtxt(data_root + 'r_s_out.txt')
    Fy = np.loadtxt(data_root + 'Fy_out.txt')
    Mz = np.loadtxt(data_root + 'Mz_out.txt')
    K, M = rs.shape
    rho = np.loadtxt(data_root + 'rho_out.txt')
    assert rho.shape[0] % K == 0
    N = rho.shape[0] // K
    rho = rho.reshape(K, N, M)
    p = np.loadtxt(data_root + 'p_out.txt').reshape(K, N, M)
    u = np.loadtxt(data_root + 'u_out.txt').reshape(K, N, M)
    v = np.loadtxt(data_root + 'v_out.txt').reshape(K, N, M)
    w = np.loadtxt(data_root + 'w_out.txt').reshape(K, N, M)

    data_dict[N_nodes_count] = (zs, rs, rho, p, u, v, w, Fy, Mz)
    print('Количество узлов:')
    print(f'z: {K}, r: {N}, theta: {M}')
    print(f'успешно загружено для {N = }')

Количество узлов:
z: 101, r: 100, theta: 300
успешно загружено для N = 100
Количество узлов:
z: 101, r: 200, theta: 600
успешно загружено для N = 200
Количество узлов:
z: 101, r: 400, theta: 1200
успешно загружено для N = 400
Количество узлов:
z: 101, r: 800, theta: 2400
успешно загружено для N = 800


In [30]:
delta_by_C_norm(data_dict[100][2][-1,:,:], data_dict[200][2][-1,::2,::2])

np.float64(0.13949999999999996)

In [42]:
delta_by_C_norm(data_dict[800][2][-1,::8,::8], data_dict[400][2][-1,::4,::4])

np.float64(0.1160000000000001)

In [36]:
delta_by_L1_norm(data_dict[100][2][-1,:,:], data_dict[200][2][-1,::2,::2], data_dict[100][1][-1,:], data_dict[100][0][-1])

np.float64(0.008480851887591024)

In [37]:
delta_by_L1_norm(data_dict[100][2][-1,:,:], data_dict[400][2][-1,::4,::4], data_dict[100][1][-1,:], data_dict[100][0][-1])

np.float64(0.011076235114796312)

In [39]:
delta_by_L1_norm(data_dict[200][2][-1,::2,::2], data_dict[400][2][-1,::4,::4], data_dict[200][1][-1,::2], data_dict[200][0][-1])

np.float64(0.003942954154378613)

In [41]:
delta_by_L1_norm(data_dict[800][2][-1,::8,::8], data_dict[400][2][-1,::4,::4], data_dict[800][1][-1,::8], data_dict[800][0][-1])

np.float64(0.0024195120613678786)